# 10강 — Spark MLlib 기반 분류/예측 모델 개발

고객 이탈(churn) 예측을 소재로, **LogisticRegression / DecisionTree / RandomForest**
세 모델을 같은 데이터로 학습·비교하고, `CrossValidator`로 하이퍼파라미터를 튜닝합니다.

학습목표 그대로: "Spark MLlib 기반 분류 및 예측 모델의 차이를 알고 개발" + "학습, 검증, 튜닝 과정을 수행"

In [ ]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.appName("mllib-classification").master("local[*]").getOrCreate()
print("SparkSession OK:", spark.version)

## 1. 실습 데이터 생성 — 고객 이탈(churn) 시뮬레이션

실제 통신사/구독 서비스 데이터를 흉내 낸 가상 데이터입니다. tenure(가입 개월 수)가 짧고, support_calls(고객센터 문의)가 많고, month-to-month 계약이면 이탈 확률이 높아지도록 신호를 심었습니다.

In [ ]:
random.seed(42)
rows = []
for i in range(2000):
    tenure = random.randint(1, 72)
    monthly_charge = round(random.uniform(20, 120), 2)
    support_calls = random.randint(0, 10)
    contract = random.choice(["month-to-month", "one-year", "two-year"])
    score = (-0.05*tenure) + (0.3*support_calls) + (10 if contract=="month-to-month" else 0) + random.gauss(0, 3)
    churn = 1 if score > 8 else 0
    rows.append((tenure, monthly_charge, support_calls, contract, churn))

df = spark.createDataFrame(rows, ["tenure", "monthly_charge", "support_calls", "contract", "churn"])
print("전체 건수:", df.count(), " 이탈 비율:", round(df.filter("churn=1").count() / df.count(), 3))
df.show(5)

## 2. 피처 엔지니어링 — 범주형 인코딩 + 벡터 조립

`contract`(범주형)를 8강에서 배운 방식대로 StringIndexer → OneHotEncoder로 변환한 뒤, 모든 피처를 하나의 벡터로 합칩니다.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

indexer = StringIndexer(inputCol="contract", outputCol="contract_idx")
encoder = OneHotEncoder(inputCols=["contract_idx"], outputCols=["contract_vec"])
assembler = VectorAssembler(
    inputCols=["tenure", "monthly_charge", "support_calls", "contract_vec"],
    outputCol="features",
)

train, test = df.randomSplit([0.8, 0.2], seed=42)
print("학습:", train.count(), " 테스트:", test.count())

## 3. 세 가지 분류 모델 학습 및 비교

같은 데이터, 같은 피처로 세 알고리즘을 각각 학습시켜 성능을 비교합니다.

- **LogisticRegression**: 선형 결정경계, 해석이 쉽고 빠름
- **DecisionTree**: 비선형 패턴을 잡지만 과적합되기 쉬움
- **RandomForest**: 여러 트리를 앙상블해 과적합을 줄임, 보통 가장 안정적

In [ ]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="churn"),
    "DecisionTree": DecisionTreeClassifier(featuresCol="features", labelCol="churn"),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="churn"),
}

auc_eval = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="churn", metricName="accuracy")

results = {}
fitted_models = {}
for name, clf in models.items():
    pipe = Pipeline(stages=[indexer, encoder, assembler, clf])
    model = pipe.fit(train)
    pred = model.transform(test)
    auc = auc_eval.evaluate(pred)
    acc = acc_eval.evaluate(pred)
    results[name] = {"AUC": round(auc, 3), "Accuracy": round(acc, 3)}
    fitted_models[name] = model
    print(f"{name:20s}  AUC={auc:.3f}  Accuracy={acc:.3f}")

### 생각해 볼 질문
1. AUC와 Accuracy 중 이탈 예측처럼 클래스가 불균형한 문제(이탈 고객이 소수)에는 어느 지표가 더 믿을 만할까요?
2. DecisionTree가 RandomForest보다 특정 지표에서 앞설 수도 있는데, 그래도 실무에서 RandomForest를 더 선호하는 이유는 무엇일까요?

## 4. 하이퍼파라미터 튜닝 — CrossValidator + ParamGridBuilder

RandomForest의 `numTrees`(트리 개수)와 `maxDepth`(트리 깊이)를 여러 조합으로 3-fold 교차검증해, 가장 좋은 조합을 자동으로 찾습니다.

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

rf = RandomForestClassifier(featuresCol="features", labelCol="churn")
pipe = Pipeline(stages=[indexer, encoder, assembler, rf])

grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 30, 50])
    .addGrid(rf.maxDepth, [3, 6, 10])
    .build()
)
print(f"탐색할 조합 수: {len(grid)}개 (numTrees 3종 x maxDepth 3종)")

cv = CrossValidator(
    estimator=pipe,
    estimatorParamMaps=grid,
    evaluator=auc_eval,
    numFolds=3,
    seed=42,
)
cv_model = cv.fit(train)

best_pred = cv_model.transform(test)
best_auc = auc_eval.evaluate(best_pred)
best_rf = cv_model.bestModel.stages[-1]
print(f"튜닝 후 테스트 AUC: {best_auc:.3f}")
print(f"최적 numTrees: {best_rf.getNumTrees}")
print(f"최적 maxDepth: {best_rf.getOrDefault('maxDepth')}")

## 5. 결과 정리

In [ ]:
import pandas as pd

summary = pd.DataFrame(results).T
summary.loc["RandomForest (튜닝 후)"] = {"AUC": round(best_auc, 3), "Accuracy": None}
print(summary)

### 생각해 볼 질문
3. 튜닝 후 AUC가 튜닝 전 RandomForest보다 별로 안 올랐다면, 그 이유로 무엇을 의심해볼 수 있을까요? (데이터 자체의 한계 vs 탐색 범위 부족)
4. `numFolds=3`을 `numFolds=5`로 늘리면 무엇이 더 좋아지고, 무엇이 더 비싸질까요?

In [ ]:
spark.stop()